In [ ]:
# Cell 1: Imports & Setup

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    accuracy_score
)

# Set seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Output settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# Ensure models directory exists
os.makedirs("models/saved", exist_ok=True)
print("[+] Environment set up successfully.")

In [ ]:
# Cell 2: Memory Reduction Function
# (Copied from fraud_detection.ipynb to ensure scalability)

def reduce_mem_usage(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        try:
            col_type = df[col].dtype

            if col_type != object and not str(col_type).startswith("category"):
                c_min = df[col].min()
                c_max = df[col].max()

                if str(col_type).startswith("int") or str(col_type).startswith("Int"):
                    if c_min >= 0:
                        if c_max < 2**8:
                            df[col] = df[col].astype(np.uint8)
                        elif c_max < 2**16:
                            df[col] = df[col].astype(np.uint16)
                        elif c_max < 2**32:
                            df[col] = df[col].astype(np.uint32)
                        else:
                            df[col] = df[col].astype(np.uint64)
                    else:
                        if np.iinfo(np.int8).min <= c_min and c_max <= np.iinfo(np.int8).max:
                            df[col] = df[col].astype(np.int8)
                        elif np.iinfo(np.int16).min <= c_min and c_max <= np.iinfo(np.int16).max:
                            df[col] = df[col].astype(np.int16)
                        elif np.iinfo(np.int32).min <= c_min and c_max <= np.iinfo(np.int32).max:
                            df[col] = df[col].astype(np.int32)
                        else:
                            df[col] = df[col].astype(np.int64)
                else:
                    df[col] = df[col].astype(np.float32)
        except Exception:
            continue

    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"[+] Mem usage decreased from {start_mem:.2f} MB to {end_mem:.2f} MB "
              f"({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)")
    return df

In [ ]:
# Cell 3: Load and Merge Data (Full Dataset)

print("[+] Loading datasets...")
try:
    df_trans = pd.read_csv('train_transaction.csv')
    df_id = pd.read_csv('train_identity.csv')
    
    print("[+] Merging datasets...")
    df = df_trans.merge(df_id, on='TransactionID', how='left')
    print(f"[+] Full dataset shape: {df.shape}")
    
    print("[+] Reducing memory usage...")
    df = reduce_mem_usage(df)
    
except FileNotFoundError:
    print("[-] Error: CSV files not found. Please ensure train_transaction.csv and train_identity.csv are in the directory.")
    raise

In [ ]:
# Cell 4: Feature Splitting & Preprocessing Setup

TARGET = 'isFraud'
if 'TransactionID' in df.columns:
    df = df.drop(columns=['TransactionID'])

y = df[TARGET]
X = df.drop(columns=[TARGET])

# Identify column types for transformer
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f"[+] Features identified: {len(num_cols)} numeric, {len(cat_cols)} categorical.")

# Define Robust Preprocessing Pipeline (Same as Isolation Forest pipeline)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())  # DBSCAN needs scaling!
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

# We will split first to avoid leakage, but PCA needs data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

print(f"[+] Train shape: {X_train.shape}")
print(f"[+] Test shape: {X_test.shape}")

In [ ]:
# Cell 5: Apply Preprocessing and PCA

# Fit preprocessor on training data
print("[+] Fitting preprocessor...")
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Apply PCA to reduce dimensionality for DBSCAN
# 434 dimensions is too high for distance-based clustering due to curse of dimensionality
N_COMPONENTS = 50 
print(f"[+] Applying PCA (reducing to {N_COMPONENTS} components)...")

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_processed)
X_test_pca = pca.transform(X_test_processed)

explained_var = pca.explained_variance_ratio_.sum()
print(f"[+] PCA Explained Variance: {explained_var:.2%}")
print(f"[+] PCA Train Shape: {X_train_pca.shape}")

In [ ]:
# Cell 6: Train DBSCAN (on Sample) and Supervised Classifier

# DBSCAN is O(N^2), so we cannot run it on 400k+ rows easily without massive RAM.
# We will use a representative subsample to find clusters, then train a KNN to generalize.

SAMPLE_SIZE = 50000
print(f"[+] Sampling {SAMPLE_SIZE} points for DBSCAN Clustering...")

# Create indexes for sampling
sample_indices = np.random.choice(X_train_pca.shape[0], SAMPLE_SIZE, replace=False)
X_sample = X_train_pca[sample_indices]

# DBSCAN Parameters
# eps: max distance between samples for them to be considered as in the same neighborhood
# min_samples: number of samples in a neighborhood for a point to be considered as a core point
# Adjusting eps is critical. With 50 PCA components, Euclidean distances are larger.
# We start with a heuristic or a tested value.
EPS = 5.0  
MIN_SAMPLES = 50

print(f"[+] Running DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})...")
dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, n_jobs=-1)
labels = dbscan.fit_predict(X_sample)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"[+] DBSCAN Complete!")
print(f"    Clusters found: {n_clusters}")
print(f"    Noise points (Anomalies): {n_noise} ({n_noise/SAMPLE_SIZE:.2%})")

# Train a Classifier to predict clusters (so we can apply this to the full dataset)
print("[+] Training KNN to generalize DBSCAN clusters to full dataset...")
knn_approximator = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_approximator.fit(X_sample, labels)

print("[+] KNN training complete.")

In [ ]:
# Cell 7: Evaluate on Test Set

print("[+] Predicting on Test Set using KNN-DBSCAN approximation...")
test_cluster_labels = knn_approximator.predict(X_test_pca)

# Map DBSCAN labels to Fraud/Normal
# -1 (Noise) -> Fraud (1)
# Anything else (Cluster) -> Normal (0)
y_pred_dbscan = (test_cluster_labels == -1).astype(int)

# Evaluate
print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred_dbscan)
print(cm)

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_dbscan))

print(f"Accuracy: {accuracy_score(y_test, y_pred_dbscan):.4f}")

# Since DBSCAN doesn't give a probability score easily, we use binary predictions for rough AUC
roc_Score = roc_auc_score(y_test, y_pred_dbscan)
print(f"ROC-AUC Score (Binary): {roc_Score:.4f}")

In [ ]:
# Cell 8: Visualization (First 2 PCA Components)

plt.figure(figsize=(10, 6))
unique_labels = set(test_cluster_labels)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

# Plot a subset of test data for visibility
subset_idx = np.random.choice(len(X_test_pca), size=min(5000, len(X_test_pca)), replace=False)
X_vis = X_test_pca[subset_idx]
y_vis = test_cluster_labels[subset_idx]
y_true_vis = y_test.iloc[subset_idx].values

# Plot Noise (Predicted Fraud) in Red, Normal Clusters in Gray/Blue
for k, col in zip(unique_labels, colors):
    if k == -1:
        # Black/Red used for noise.
        col = [1, 0, 0, 1] # Red
        alpha = 0.6
        label = "Predicted Anomaly"
    else:
        col = [0.5, 0.5, 0.5, 0.2] # Gray, transparent
        alpha = 0.1
        label = "Predicted Normal" if k == 0 else ""

    class_member_mask = (y_vis == k)
    xy = X_vis[class_member_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=4, alpha=alpha, label=label)

plt.title('DBSCAN Clustering Results (PCA Projection)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Cell 9: Save Updated Models

print("[+] Saving Scaler/PCA/KNN models...")
joblib.dump(preprocessor, 'models/saved/full_preprocessor.pkl')
joblib.dump(pca, 'models/saved/pca_50.pkl')
joblib.dump(knn_approximator, 'models/saved/knn_dbscan_proxy.pkl')
print("[+] Models saved successfully.")